# Eksplor isi `data_multilevel.grib`

Notebook ini hanya membaca header GRIB untuk melihat isi file: variabel, level tekanan, waktu, dan domain grid. Tidak ada plot dan tidak ada ekstraksi payload data.

In [ ]:
from __future__ import annotations

from pathlib import Path
from zipfile import is_zipfile

import pandas as pd
from IPython.display import display
from eccodes import codes_get, codes_grib_new_from_file, codes_release

DATA_PATH = Path("/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/era5-monthly/data_multilevel.grib")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

assert DATA_PATH.exists(), f"File tidak ditemukan: {DATA_PATH}"

In [ ]:
def yyyymmdd_to_label(value: int) -> str:
    text = f"{int(value):08d}"
    return f"{text[:4]}-{text[4:6]}-{text[6:8]}"


def scan_grib_headers(path: Path) -> pd.DataFrame:
    keys = [
        "shortName",
        "name",
        "paramId",
        "typeOfLevel",
        "level",
        "dataDate",
        "dataTime",
        "gridType",
        "Ni",
        "Nj",
        "latitudeOfFirstGridPointInDegrees",
        "longitudeOfFirstGridPointInDegrees",
        "latitudeOfLastGridPointInDegrees",
        "longitudeOfLastGridPointInDegrees",
        "iDirectionIncrementInDegrees",
        "jDirectionIncrementInDegrees",
    ]

    rows = []
    with path.open("rb") as fh:
        while True:
            gid = codes_grib_new_from_file(fh)
            if gid is None:
                break
            rows.append({key: codes_get(gid, key) for key in keys})
            codes_release(gid)

    return pd.DataFrame(rows)

In [ ]:
headers = scan_grib_headers(DATA_PATH)
first_row = headers.iloc[0].to_dict()
last_row = headers.iloc[-1].to_dict()

file_summary = pd.DataFrame(
    [
        {
            "file": DATA_PATH.name,
            "size_mb": round(DATA_PATH.stat().st_size / (1024 * 1024), 2),
            "is_zipfile": is_zipfile(DATA_PATH),
            "messages": len(headers),
            "unique_variables": headers["shortName"].nunique(),
            "unique_levels": headers["level"].nunique(),
            "unique_dates": headers["dataDate"].nunique(),
            "first_date": yyyymmdd_to_label(headers["dataDate"].min()),
            "last_date": yyyymmdd_to_label(headers["dataDate"].max()),
            "monthly_start_dates": headers["dataDate"].astype(str).str.endswith("01").all(),
            "time_is_00UTC": headers["dataTime"].eq(0).all(),
        }
    ]
)

display(file_summary)

In [ ]:
variable_summary = (
    headers.groupby("shortName", as_index=False)
    .agg(
        name=("name", "first"),
        paramId=("paramId", "first"),
        messages=("shortName", "size"),
        level_count=("level", "nunique"),
        levels=("level", lambda s: sorted(s.unique().tolist())),
    )
    .sort_values("shortName")
    .reset_index(drop=True)
)

display(variable_summary)

In [ ]:
unique_dates = sorted(headers["dataDate"].unique().tolist())

time_summary = pd.DataFrame(
    [
        {"metric": "first_date", "value": yyyymmdd_to_label(unique_dates[0])},
        {"metric": "last_date", "value": yyyymmdd_to_label(unique_dates[-1])},
        {"metric": "unique_months", "value": len(unique_dates)},
        {"metric": "day_of_month_is_01", "value": headers["dataDate"].astype(str).str.endswith("01").all()},
        {"metric": "time_is_00UTC", "value": headers["dataTime"].eq(0).all()},
        {"metric": "cadence_note", "value": "Bulanan, tanggal 01 pukul 00:00 UTC"},
    ]
)

print("Daftar awal bulan:", [yyyymmdd_to_label(d) for d in unique_dates[:12]])
print("Daftar akhir bulan:", [yyyymmdd_to_label(d) for d in unique_dates[-12:]])
display(time_summary)

In [ ]:
domain_cols = [
    "gridType",
    "Ni",
    "Nj",
    "latitudeOfFirstGridPointInDegrees",
    "longitudeOfFirstGridPointInDegrees",
    "latitudeOfLastGridPointInDegrees",
    "longitudeOfLastGridPointInDegrees",
    "iDirectionIncrementInDegrees",
    "jDirectionIncrementInDegrees",
]

domain_summary = pd.DataFrame(
    [
        {
            "gridType": headers["gridType"].iloc[0],
            "Ni": int(headers["Ni"].iloc[0]),
            "Nj": int(headers["Nj"].iloc[0]),
            "lat_first": headers["latitudeOfFirstGridPointInDegrees"].iloc[0],
            "lon_first": headers["longitudeOfFirstGridPointInDegrees"].iloc[0],
            "lat_last": headers["latitudeOfLastGridPointInDegrees"].iloc[0],
            "lon_last": headers["longitudeOfLastGridPointInDegrees"].iloc[0],
            "i_increment": headers["iDirectionIncrementInDegrees"].iloc[0],
            "j_increment": headers["jDirectionIncrementInDegrees"].iloc[0],
        }
    ]
)

domain_checks = pd.DataFrame(
    [
        {
            "key": col,
            "unique_values": headers[col].nunique(),
            "example": headers[col].iloc[0],
        }
        for col in domain_cols
    ]
)

display(domain_summary)
display(domain_checks)

In [ ]:
sample_cols = ["shortName", "name", "typeOfLevel", "level", "dataDate", "dataTime", "gridType", "Ni", "Nj"]

print("Dua belas message awal:")
display(headers.head(12)[sample_cols])

print("Dua belas message akhir:")
display(headers.tail(12)[sample_cols])

print("Message pertama:")
print(first_row)

print("Message terakhir:")
print(last_row)

## Next processing options

- Buka file ini dengan `cfgrib` lalu pilih variabel tertentu memakai `filter_by_keys` untuk `shortName` dan `typeOfLevel`.
- Rename koordinat ke skema yang dipakai notebook lain: `time`, `lat`, `lon`, dan `level`.
- Subset ke domain target dan level tekanan yang dibutuhkan sebelum membangun dataset analisis korelasi.
- Simpan hasil turunan ke NetCDF atau Zarr agar pembacaan berikutnya lebih cepat.

In [ ]:
# Export sekali, lalu pecah menjadi file NetCDF terpisah per variabel.
# u dan v digabung dalam satu file karena biasanya dipakai bersama sebagai komponen angin.
import xarray as xr

OUTPUT_DIR = DATA_PATH.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def standardize_coords(ds: xr.Dataset) -> xr.Dataset:
    rename_map = {}
    for old, new in (("latitude", "lat"), ("longitude", "lon"), ("isobaricInhPa", "level")):
        if old in ds.dims or old in ds.coords:
            rename_map[old] = new
    if rename_map:
        ds = ds.rename(rename_map)
    return ds


def build_encoding(ds: xr.Dataset) -> dict[str, dict[str, object]]:
    encoding: dict[str, dict[str, object]] = {}
    for var_name, da in ds.data_vars.items():
        chunksizes = []
        for dim in da.dims:
            size = ds.sizes[dim]
            if dim == "time":
                chunksizes.append(min(12, size))
            elif dim == "level":
                chunksizes.append(1)
            elif dim == "lat":
                chunksizes.append(min(181, size))
            elif dim == "lon":
                chunksizes.append(min(360, size))
            else:
                chunksizes.append(size)
        encoding[var_name] = {
            "zlib": True,
            "complevel": 4,
            "chunksizes": tuple(chunksizes),
        }
    return encoding


ds = xr.open_dataset(
    DATA_PATH,
    engine="cfgrib",
    backend_kwargs={"indexpath": ""},
    chunks={"time": 12},
)
ds = standardize_coords(ds)

exports = [
    (["u", "v"], "uv_wind_1980-2025_1000hpa-100hpa.nc"),
    (["w"], "vertical_velocity_1980-2025_1000hpa-100hpa.nc"),
    (["z"], "geopotential_1980-2025_1000hpa-100hpa.nc"),
    (["q"], "specific_humidity_1980-2025_1000hpa-100hpa.nc"),
    (["t"], "temperature_1980-2025_1000hpa-100hpa.nc"),
    (["vo"], "relative_vorticity_1980-2025_1000hpa-100hpa.nc"),
]

written = []
for var_names, filename in exports:
    missing = [name for name in var_names if name not in ds.data_vars]
    if missing:
        raise KeyError(f"Variabel tidak ditemukan di dataset: {missing}")
    out_path = OUTPUT_DIR / filename
    subset = ds[var_names]
    subset.to_netcdf(out_path, engine="netcdf4", encoding=build_encoding(subset))
    written.append(out_path)

print("File NetCDF tertulis:")
for path in written:
    print(path)

# Saran praktis:
# - Kalau nanti sering baca subset kecil atau bikin banyak analisis ulang, Zarr biasanya lebih cepat dan lebih cocok daripada NetCDF.
# - Kalau tetap NetCDF, pola satu variabel satu file + chunking seperti di atas sudah jauh lebih enak daripada buka GRIB berulang.


In [ ]:
#open w but without decode times
w = xr.open_dataset('/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/era5-monthly/w_vertical_velocity_1980-2025_1000hpa-100hpa.nc', chunks={'time': 12, 'level': 1, 'lat': 181, 'lon': 360}, decode_times=False).drop_vars('step')
